# IBM员工离职分析与可视化
模拟企业HR场景：从岗位诊断、员工分群到离职识别和资源安排。

数据为附件A明确对应的IBM HR虚构样本；不是IBM真实员工数据。全流程不联网、不安装库。请从本目录启动Jupyter。

说明：发布版清除了本地执行输出，请按顺序运行分析单元。绘图单元会从数据重建最终展示样式，与命令行使用同一入口。

In [ ]:
from pathlib import Path
import sys, os, json
ROOT=Path.cwd()
assert (ROOT/'scripts/pipeline.py').exists(), '请在项目根目录运行Notebook'
os.environ['OMP_NUM_THREADS']='1'
sys.path.insert(0,str(ROOT/'scripts'))
import pandas as pd
from IPython.display import display, Image
import pipeline, visuals
print('Python:',sys.executable)

## 1 数据与来源核对
通过官方文件下载API获取；原文件、说明、许可和哈希均保留。

In [ ]:
display(json.loads((ROOT/'data/raw/manifest.json').read_text(encoding='utf-8')))
raw=pd.read_csv(ROOT/'data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print('行列数:',raw.shape,'缺失:',raw.isna().sum().sum(),'重复:',raw.duplicated().sum())
display(raw.head())

## 2 重新计算三个模型模块
ABC用于岗位离职案例贡献；分群不使用目标标签；分类先分测试集，交叉验证和阈值仅用训练集。此单元格执行完整计算。

In [ ]:
pipeline.run()

## 3 岗位ABC结果

In [ ]:
display(pd.read_csv(ROOT/'outputs/tables/abc_roles.csv'))
display(pd.read_csv(ROOT/'outputs/tables/abc_sensitivity.csv'))

## 4 员工结构与群数选择

In [ ]:
display(pd.read_csv(ROOT/'outputs/tables/cluster_selection.csv'))
display(pd.read_csv(ROOT/'outputs/tables/cluster_profile.csv'))
display(pd.read_csv(ROOT/'outputs/tables/cluster_outcomes.csv'))

## 5 分类评估
不同阈值的召回率不能直接当作算法优劣；优先比较AP/AUC。误报是需要管理资源承担的代价。

In [ ]:
display(pd.read_csv(ROOT/'outputs/tables/cv_scores.csv'))
display(pd.read_csv(ROOT/'outputs/tables/test_metrics.csv'))
display(pd.read_csv(ROOT/'outputs/tables/test_bootstrap_ci.csv'))
display(pd.read_csv(ROOT/'outputs/tables/confusion.csv'))

## 6 生成24张图并查看核心结果

In [ ]:
visuals.run()
import runpy
runpy.run_path(str(ROOT/"scripts/dashboard.py"))
index=pd.read_csv(ROOT/'outputs/tables/figure_index.csv')
display(index[['编号','对应模板或模型']])
for number in ['fig11','fig16','fig18','fig19']:
    name=index.loc[index['编号']==number,'文件'].iloc[0]
    display(Image(filename=str(ROOT/'outputs/figures'/name),width=700))

## 7 大屏与报告
双击 `outputs/dashboard/员工离职管理.html` 即可离线交互，不需要Python服务器。也可以运行 `python 启动大屏.py`。

报告入口：`docs/商业数据分析期末论文.docx`。经营建议见第六章，实施计划按模拟场景假设设计，未宣称已经取得留人效果。仓库：https://github.com/ccy-3927/business-data-analysis-hr